In [2]:
queries = [
    "AdTech", 
    "CRM", 
    "AI Labs", 
    "FinTech", 
    "Cybersecurity", 
    "LegalTech", 
    "EdTech", 
    "HealthTech", 
    "PropTech", 
    "MarTech", 
    "HRTech", 
    "RetailTech", 
    "TravelTech", 
    "AgriTech", 
    "BioTech", 
    "SportsTech", 
    "InsurTech", 
    "CleanTech", 
    "Gaming", 
    "MediaTech"
]

In [3]:
len(queries)

20

## dataset curation

In [2]:
!pip3 install openai

In [4]:
import sys
import os

# Add the path to the parent directory
sys.path.append(os.path.abspath(".."))

# Now you can import the module
from dep.openai_api import OpenAIChatCompletion
from dep.response_formats import SyntheticPositiveFactsheetResponse

In [5]:
!pwd

/home/ec2-user/SageMaker/ColBERT


In [6]:
from dep.response_formats import SyntheticPositiveFactsheetResponse
response_format=SyntheticPositiveFactsheetResponse

In [41]:
# res=llm.get_text_completion(prompt,0)['choices'][0]['message']['content']
def generate_factsheets(query,n_fatchsheets=20):
    success=False
    
    sys_msg_path1='../prompts/synthetic_positive_factsheet.txt'
    system_message1=open(sys_msg_path1,'r').read()
    llm1=OpenAIChatCompletion(model='gpt-4o',system_message=system_message1)
    
    sys_msg_path2='../prompts/v0.txt'
    system_message2=open(sys_msg_path2,'r').read()
    llm2=OpenAIChatCompletion(model='gpt-4o',system_message=system_message2)

    
    prompt1=f"Please provide {2*n_fatchsheets//3} facsheets for companies that are positives for the query: {query}"
    
    prompt2=f"Please provide {n_fatchsheets//3} facsheets for companies that are positives for the query: {query}"

    temperature=0.3
    
    res1=llm1.structured_response(prompt1,temperature,response_format)
    res2=llm1.structured_response(prompt2,temperature,response_format)
    try:
        r1=json.loads(res1)['factsheets']
        r2=json.loads(res2)['factsheets']

        success=True
        return success,r1+r2
    except Exception as e:
        
        print(f'failed for {query} - {e}')
        success=False
        return success,[]
    

In [42]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import tqdm.notebook as tqdm
import json
def generate_factsheets_parallel(queries, n_factsheets=20, max_workers=10):
    """
    Run generate_factsheets in parallel for multiple queries.
    
    Args:
        queries (list): List of query strings to process
        n_factsheets (int): Number of factsheets to generate per query
        max_workers (int): Maximum number of parallel threads
        
    Returns:
        dict: Dictionary mapping queries to (success, factsheets) tuples
    """
    results = {}
    
    # Use tqdm for progress bar in Jupyter
    progress_bar = tqdm.tqdm(total=len(queries), desc="Processing queries")
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Start the tasks
        future_to_query = {
            executor.submit(generate_factsheets, query, n_factsheets): query 
            for query in queries
        }
        
        # Process results as they complete
        for future in as_completed(future_to_query):
            query = future_to_query[future]
            try:
                success, factsheets = future.result()
                results[query] = (success, factsheets)
            except Exception as e:
                print(f"Query '{query}' generated an exception: {e}")
                results[query] = (False, [])
            
            progress_bar.update(1)
    
    progress_bar.close()
    return results

In [25]:
queries_to_generate_factsheets=[('Consumer Industry',query) for query in queries] + [('Industry',query) for query in queries]

In [ ]:
generate_factsheets_parallel(queries_to_generate_factsheets[:6],n_factsheets=20,max_workers=3)

In [27]:
train_results=generate_factsheets_parallel(queries_to_generate_factsheets,n_factsheets=20,max_workers=20)

Processing queries:   0%|          | 0/40 [00:00<?, ?it/s]

In [37]:
import pickle
with open('../synthetic_data/synthetic_factsheets.train.pkl','wb') as f:
    pickle.dump(train_results,f)

In [43]:
test_results=generate_factsheets_parallel(queries_to_generate_factsheets,n_factsheets=6,max_workers=20)

Processing queries:   0%|          | 0/40 [00:00<?, ?it/s]

In [44]:
with open('../synthetic_data/synthetic_factsheets.test.pkl','wb') as f:
    pickle.dump(test_results,f)

In [112]:
val_results=generate_factsheets_parallel(queries_to_generate_factsheets,n_factsheets=6,max_workers=20)

Processing queries:   0%|          | 0/40 [00:00<?, ?it/s]

In [113]:
with open('../synthetic_data/synthetic_factsheets.val.pkl','wb') as f:
    pickle.dump(val_results,f)

In [47]:
import random
from typing import Dict, List, Tuple, Any

def create_training_triplets(
    factsheet_dict: Dict[Tuple[str, str], Tuple[bool, List[str]]],
    k: int = 5,
    strategy1_proportion: float = 0.5,
    random_seed: int = 42
) -> List[Tuple[Tuple[str, str], str, str]]:
    """
    Create training triplets from factsheet dictionary.
    
    Args:
        factsheet_dict: Dictionary mapping (query_type, query_value) to (success, [factsheets])
        k: Number of negatives to sample per positive
        strategy1_proportion: Proportion of negatives to sample using strategy 1 
            (0.0 = all strategy 2, 1.0 = all strategy 1)
        random_seed: Seed for reproducibility
        
    Returns:
        List of triplets in format [(query_tuple, positive_doc, negative_doc)]
    """
    random.seed(random_seed)
    
    # Extract successful factsheets only
    successful_factsheets = {
        query: docs for query, (success, docs) in factsheet_dict.items() 
        if success and docs
    }
    
    # Organize by query type and value for easier access
    industry_factsheets = {}
    consumer_industry_factsheets = {}
    
    for (query_type, query_value), docs in successful_factsheets.items():
        if query_type == "Industry":
            if query_value not in industry_factsheets:
                industry_factsheets[query_value] = []
            industry_factsheets[query_value].extend(docs)
        elif query_type == "Consumer Industry":
            if query_value not in consumer_industry_factsheets:
                consumer_industry_factsheets[query_value] = []
            consumer_industry_factsheets[query_value].extend(docs)
    
    triplets = []
    
    # Create triplets from Industry queries
    for query_value, positives in industry_factsheets.items():
        query_tuple = ("Industry", query_value)
        
        for positive in positives:
            # Calculate how many negatives to sample from each strategy
            strategy1_count = int(k * strategy1_proportion)
            strategy2_count = k - strategy1_count
            
            # Strategy 1: Use (Consumer Industry, same_query) as negatives
            strategy1_negatives = []
            if strategy1_count > 0 and query_value in consumer_industry_factsheets:
                # Sample from Consumer Industry with same query value
                strategy1_candidates = consumer_industry_factsheets[query_value]
                strategy1_negatives = random.sample(
                    strategy1_candidates, 
                    min(strategy1_count, len(strategy1_candidates))
                )
                
            # Strategy 2: Use (Industry, different_query) as negatives
            strategy2_negatives = []
            if strategy2_count > 0:
                # Collect all factsheets from other Industry queries
                strategy2_candidates = []
                for other_query, other_docs in industry_factsheets.items():
                    if other_query != query_value:
                        strategy2_candidates.extend(other_docs)
                
                if strategy2_candidates:
                    strategy2_negatives = random.sample(
                        strategy2_candidates,
                        min(strategy2_count, len(strategy2_candidates))
                    )
            
            # Combine negatives from both strategies
            negatives = strategy1_negatives + strategy2_negatives
            
            # Create triplets
            for negative in negatives:
                triplets.append((query_tuple, positive, negative))
    
    # Create triplets from Consumer Industry queries
    for query_value, positives in consumer_industry_factsheets.items():
        query_tuple = ("Consumer Industry", query_value)
        
        for positive in positives:
            # Calculate how many negatives to sample from each strategy
            strategy1_count = int(k * strategy1_proportion)
            strategy2_count = k - strategy1_count
            
            # Strategy 1: Use (Industry, same_query) as negatives
            strategy1_negatives = []
            if strategy1_count > 0 and query_value in industry_factsheets:
                # Sample from Industry with same query value
                strategy1_candidates = industry_factsheets[query_value]
                strategy1_negatives = random.sample(
                    strategy1_candidates,
                    min(strategy1_count, len(strategy1_candidates))
                )
                
            # Strategy 2: Use (Consumer Industry, different_query) as negatives
            strategy2_negatives = []
            if strategy2_count > 0:
                # Collect all factsheets from other Consumer Industry queries
                strategy2_candidates = []
                for other_query, other_docs in consumer_industry_factsheets.items():
                    if other_query != query_value:
                        strategy2_candidates.extend(other_docs)
                
                if strategy2_candidates:
                    strategy2_negatives = random.sample(
                        strategy2_candidates,
                        min(strategy2_count, len(strategy2_candidates))
                    )
            
            # Combine negatives from both strategies
            negatives = strategy1_negatives + strategy2_negatives
            
            # Create triplets
            for negative in negatives:
                triplets.append((query_tuple, positive, negative))
    
    # Shuffle the triplets
    random.shuffle(triplets)
    
    return triplets

# Example usage:
# triplets = create_training_triplets(factsheet_dict, k=5, strategy1_proportion=0.6)

In [52]:
training_triplets=create_training_triplets(
    factsheet_dict=train_results,
    k= 20,
    strategy1_proportion=0.7,
    random_seed = 42
) 

In [114]:
val_triplets=create_training_triplets(
factsheet_dict=val_results,
k= 6,
strategy1_proportion=0.6,
random_seed = 42
)

In [148]:
test_triples=create_training_triplets(
factsheet_dict=test_results,
k= 6,
strategy1_proportion=0.6,
random_seed = 42
)

In [115]:
with open('../synthetic_data/triples.val.pkl','wb') as f:
    pickle.dump(val_triplets,f)

In [55]:
with open('../synthetic_data/triples.train.pkl','wb') as f:
    pickle.dump(training_triplets,f)

In [188]:
with open('../synthetic_data/triples.test.pkl','wb') as f:
    pickle.dump(test_triples,f)

In [91]:
import pandas as pd
training_triplets_df=pd.DataFrame(training_triplets,columns=['query','collection','negative'])


In [92]:
training_triplets_df['nlq']=training_triplets_df['query'].apply(lambda x: " - ".join(x))

In [93]:
train_queries=training_triplets_df[['nlq']].drop_duplicates()
train_collection=pd.concat([training_triplets_df[['collection']].drop_duplicates(),training_triplets_df[['negative']].rename(columns={'negative':'collection'}).drop_duplicates()]).drop_duplicates()



In [94]:
train_queries['qid']=list(range(len(train_queries)))
train_collection['id']=list(range(len(train_collection)))

In [116]:
val_triplets_df=pd.DataFrame(val_triplets,columns=['query','collection','negative'])
val_triplets_df['nlq']=val_triplets_df['query'].apply(lambda x: " - ".join(x))

val_queries=val_triplets_df[['nlq']].drop_duplicates()
val_collection=pd.concat([val_triplets_df[['collection']].drop_duplicates(),val_triplets_df[['negative']].rename(columns={'negative':'collection'}).drop_duplicates()]).drop_duplicates()


val_queries['qid']=list(range(len(val_queries)))
val_collection['id']=list(range(len(val_collection)))

In [117]:
val_collection

,collection,id
0,**Company Name:** LearnTech Innovations \n**W...,0
1,**Company Name:** DataBridge Solutions \n**We...,1
2,\n**Company Name:** CustomerConnect Innovation...,2
3,**Company Name:** FinEdge Innovations \n**Web...,3
4,**Company Name:** EduFuture Technologies \n**...,4
...,...,...
743,**Company Name:** VirtualRealms Entertainment ...,234
768,**Company Name:** AIForge Technologies \n**We...,235
815,**Company Name:** TalentSync Solutions \n**We...,236
888,**Company Name:** Synapse AI Research \n**Web...,237


In [95]:
train_collection

,collection,id
0,**Company Name:** MedLink Solutions \n**Websi...,0
1,**Company Name:** MedTech Analytics \n**Websi...,1
2,**Company Name:** PayTech Innovations \n**Web...,2
3,**Company Name:** AudioWave Innovations \n**W...,3
4,**Company Name:** LawSync Technologies \n**We...,4
...,...,...
3116,**Company Name:** GeoTherm Solutions \n**Webs...,752
3231,**Company Name:** GreenTech Innovations \n**W...,753
3350,**Company Name:** BioTech Manufacturing Soluti...,754
3796,**Company Name:** FarmNet Solutions \n**Websi...,755


In [96]:
train_queries

,nlq,qid
0,Consumer Industry - HealthTech,0
2,Industry - FinTech,1
3,Industry - MediaTech,2
4,Consumer Industry - LegalTech,3
5,Industry - HealthTech,4
7,Industry - PropTech,5
8,Industry - InsurTech,6
9,Consumer Industry - BioTech,7
10,Industry - AgriTech,8
11,Industry - MarTech,9


In [97]:
training_triplets_df=training_triplets_df.merge(train_queries,how='left')

In [98]:
training_triplets_df=training_triplets_df.merge(train_collection.rename(columns={'id':'pid'}),how='left',on='collection')
training_triplets_df=training_triplets_df.merge(train_collection.rename(columns={'id':'nid','collection':'negative'}),how='left',on='negative')

In [118]:
val_triplets_df=val_triplets_df.merge(val_queries,how='left')
val_triplets_df=val_triplets_df.merge(val_collection.rename(columns={'id':'pid'}),how='left',on='collection')
val_triplets_df=val_triplets_df.merge(val_collection.rename(columns={'id':'nid','collection':'negative'}),how='left',on='negative')

In [99]:
training_triplets_df

,query,collection,negative,nlq,qid,pid,nid
0,"(Consumer Industry, HealthTech)",**Company Name:** MedLink Solutions \n**Websi...,**Company Name:** NeuroTech Labs \n**Website:...,Consumer Industry - HealthTech,0,0,161
1,"(Consumer Industry, HealthTech)",**Company Name:** MedTech Analytics \n**Websi...,**Company Name:** BioTech Innovations \n**Web...,Consumer Industry - HealthTech,0,1,89
2,"(Industry, FinTech)",**Company Name:** PayTech Innovations \n**Web...,**Company Name:** PayLink Innovations \n**Web...,Industry - FinTech,1,2,324
3,"(Industry, MediaTech)",**Company Name:** AudioWave Innovations \n**W...,**Company Name:** AudioWave Technologies \n**...,Industry - MediaTech,2,3,560
4,"(Consumer Industry, LegalTech)",**Company Name:** LawSync Technologies \n**We...,**Company Name:** JurisTech Solutions \n**Web...,Consumer Industry - LegalTech,3,4,563
...,...,...,...,...,...,...,...
15155,"(Industry, MarTech)",**Company Name:** EngageTech Systems \n**Webs...,**Company Name:** EngageTech Solutions \n**We...,Industry - MarTech,9,505,117
15156,"(Industry, LegalTech)",**Company Name:** LegalVision Technologies \n...,**Company Name:** LawBridge Solutions \n**Web...,Industry - LegalTech,34,648,317
15157,"(Consumer Industry, AgriTech)",**Company Name:** FarmTech Systems \n**Websit...,**Company Name:** AdConnect Systems \n**Websi...,Consumer Industry - AgriTech,13,646,129
15158,"(Industry, EdTech)",**Company Name:** LearnQuest Technologies \n*...,**Company Name:** LearnSync Solutions \n**Web...,Industry - EdTech,17,733,410


In [100]:
train_collection['collection']=train_collection['collection'].apply(lambda x:x.replace('\t',' ').replace('\n',' '))

In [101]:
train_collection[['id','collection']].to_csv('../synthetic_data/corpus.train.tsv',header=None,index=None, sep='\t')

In [119]:
val_collection['collection']=val_collection['collection'].apply(lambda x:x.replace('\t',' ').replace('\n',' '))
val_collection[['id','collection']].to_csv('../synthetic_data/corpus.val.tsv',header=None,index=None, sep='\t')

In [102]:
pd.read_csv('../synthetic_data/corpus.train.tsv',header=None,index_col=None, sep='\t')

,0,1
0,0,**Company Name:** MedLink Solutions **Websit...
1,1,**Company Name:** MedTech Analytics **Websit...
2,2,**Company Name:** PayTech Innovations **Webs...
3,3,**Company Name:** AudioWave Innovations **We...
4,4,**Company Name:** LawSync Technologies **Web...
...,...,...
752,752,**Company Name:** GeoTherm Solutions **Websi...
753,753,**Company Name:** GreenTech Innovations **We...
754,754,**Company Name:** BioTech Manufacturing Soluti...
755,755,**Company Name:** FarmNet Solutions **Websit...


In [120]:
pd.read_csv('../synthetic_data/corpus.val.tsv',header=None,index_col=None, sep='\t')

,0,1
0,0,**Company Name:** LearnTech Innovations **We...
1,1,**Company Name:** DataBridge Solutions **Web...
2,2,**Company Name:** CustomerConnect Innovations...
3,3,**Company Name:** FinEdge Innovations **Webs...
4,4,**Company Name:** EduFuture Technologies **W...
...,...,...
234,234,**Company Name:** VirtualRealms Entertainment ...
235,235,**Company Name:** AIForge Technologies **Web...
236,236,**Company Name:** TalentSync Solutions **Web...
237,237,**Company Name:** Synapse AI Research **Webs...


In [103]:
train_queries[['qid','nlq']].to_csv('../synthetic_data/queries.train.tsv',header=None,index=None, sep='\t')

In [121]:
val_queries[['qid','nlq']].to_csv('../synthetic_data/queries.val.tsv',header=None,index=None, sep='\t')

In [142]:
import numpy as np
def save_triplets_for_training(
    triplets_df: pd.DataFrame,
    output_file: str
) -> None:
    """
    Save triplets to JSONL format expected by the training module.
    Each line is a JSON array with [qid, pid, nid]
    
    Args:
        triplets_df: DataFrame with columns 'qid', 'pid', and 'nid'
        output_file: Path to output JSONL file
    """
    # Create directory if it doesn't exist
    os.makedirs(os.path.dirname(os.path.abspath(output_file)), exist_ok=True)
    
    with open(output_file, 'w', encoding='utf-8') as f:
        for _, row in triplets_df.iterrows():
            # Convert values to native Python types to ensure JSON serialization works
            example = [
                int(row['qid']) if isinstance(row['qid'], (int, float,np.int64)) else int(row['qid']),
                int(row['pid']) if isinstance(row['pid'], (int, float,np.int64)) else int(row['pid']),
                int(row['nid']) if isinstance(row['nid'], (int, float,np.int64)) else int(row['nid'])
            ]
            # print(f'example id qid type {type(row["qid"])} , {isinstance(row["qid"], (int, float,np.int64))}, {type(example[0])}')
            # Write JSON array to file
            f.write(ujson.dumps(example) + '\n')
    
    print(f"Saved {len(triplets_df)} triplets to {output_file}")

In [141]:
import ujson
save_triplets_for_training(training_triplets_df[['qid','pid','nid']],'../synthetic_data/triples.train.jsonl')

example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'

In [143]:
save_triplets_for_training(val_triplets_df[['qid','pid','nid']],'../synthetic_data/triples.val.jsonl')

Saved 1440 triplets to ../synthetic_data/triples.val.jsonl


In [139]:
save_triplets_for_training(val_triplets_df[['qid','pid','nid']],'../synthetic_data/triples.random.jsonl')

example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'> , True, <class 'int'>
example id qid type <class 'numpy.int64'

In [124]:
# val_triplets_df

## evaluating

In [144]:

from colbert.modeling.checkpoint import Checkpoint
from colbert.infra import ColBERTConfig
from colbert.modeling.colbert import colbert_score

checkpoint = './experiments/colbert_aspect_training/none/run_1741616267/checkpoints/colbert-best'
checkpoint = "./experiments/colbert_aspect_training/none/run_1741795018/checkpoints/colbert-7000"
checkpoint = './experiments/colbert_aspect_training/train_colbert/run_1742250943/checkpoints/colbert-175'

config = ColBERTConfig(
                bsize=64,  # Small batch size for testing
                accumsteps=2,
                lr=5e-6,
                nway=2,  # Binary pairs for simplicity  
                query_maxlen=32,  
                doc_maxlen=512,   
                dim=128,
                similarity="cosine",
                use_ib_negatives=False,
                maxsteps=10000,  # Limit training steps
                warmup=250,
                val_check_interval=150,
                val_ema_alpha=0.95,
                attend_to_mask_tokens=True
            )




import torch
import numpy as np
ckpt = Checkpoint(checkpoint, colbert_config=config)


/home/ec2-user/SageMaker/ColBERT/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()


In [151]:
test_triples_df=pd.DataFrame(test_triples,columns=['query','collection','negative'])


In [153]:
test_triples_df['nlq']=test_triples_df['query'].apply(lambda x:' - '.join(x))

In [155]:
test_positives_df=test_triples_df[['query','nlq','collection']].drop_duplicates()

In [156]:
test_positives_df

,query,nlq,collection
0,"(Industry, EdTech)",Industry - EdTech,**Company Name:** LearnTech Innovations \n**W...
1,"(Consumer Industry, InsurTech)",Consumer Industry - InsurTech,**Company Name:** DataGuard Solutions \n**Web...
2,"(Industry, FinTech)",Industry - FinTech,**Company Name:** NextGen Payments \n**Websit...
3,"(Industry, MarTech)",Industry - MarTech,**Company Name:** MarTech Innovators Inc. \n*...
4,"(Industry, EdTech)",Industry - EdTech,**Company Name:** EduFuture Technologies \n**...
...,...,...,...
743,"(Industry, Gaming)",Industry - Gaming,**Company Name:** IndieQuest Games \n**Websit...
768,"(Consumer Industry, AI Labs)",Consumer Industry - AI Labs,**Company Name:** InnovateAI Labs \n**Website...
815,"(Industry, LegalTech)",Industry - LegalTech,**Company Name:** LexiTech Innovations \n**We...
888,"(Industry, Cybersecurity)",Industry - Cybersecurity,**Company Name:** ShieldTech Innovations \n**...


In [157]:
industry_test_subset_df=test_positives_df[~test_positives_df.nlq.str.contains('Consumer Industry -')]
consumer_test_subset_df=test_positives_df[test_positives_df.nlq.str.contains('Consumer Industry -')]

In [183]:
test_query='Online Marketplaces'
test_query='Electric Utilities'
test_query='FinTech'
test_query='AI Labs'
test_query='Gaming'
query2=f'Consumer Industry - {test_query}'
query1=f'Industry - {test_query}'


pos_for_1=industry_test_subset_df[industry_test_subset_df['nlq']==query1].collection.sample(5).to_list()
pos_for_2=consumer_test_subset_df[consumer_test_subset_df['nlq']==query2].collection.sample(5).to_list()

In [194]:
queries=[query1,query2]
data=pos_for_1+pos_for_2
query_score_dict={}
query_rank_dict={}

for query in queries:
    Q = ckpt.queryFromText([query])
    D = ckpt.docFromText(data, bsize=1024)[0]
    D_mask = torch.ones(D.shape[:2], dtype=torch.long)
    scores = colbert_score(Q, D, D_mask).flatten().cpu().numpy().tolist()
    query_score_dict[query]=scores
    ranking_colbert = np.argsort(scores)[::-1]
    query_rank_dict[query]=ranking_colbert

In [195]:
query_rank_dict

{'Industry - Gaming': array([0, 2, 1, 9, 4, 3, 8, 7, 5, 6]),
 'Consumer Industry - Gaming': array([7, 5, 6, 8, 9, 3, 4, 1, 0, 2])}

In [196]:
query_score_dict

{'Industry - Gaming': [16.359375,
  14.5859375,
  15.3125,
  9.390625,
  10.7578125,
  6.8203125,
  5.515625,
  7.26171875,
  8.7265625,
  11.3125],
 'Consumer Industry - Gaming': [10.0390625,
  11.6875,
  9.859375,
  12.3359375,
  11.8203125,
  14.0625,
  13.953125,
  14.625,
  13.7421875,
  12.65625]}

In [186]:
pos_for_1

['**Company Name:** GameCraft Innovations  \n**Website:** gamecraftinnovations.com  \n**Industry:** Specializes in the creation of mobile and console games with a focus on multiplayer online battle arenas (MOBAs) and esports.  \n**Target Audience:** Esports leagues, gaming influencers, and streaming platforms looking for competitive gaming content.  \n**Products & Services:**  \n- **BattleZone:** A flagship MOBA game known for its strategic depth and competitive balance.  \n- **Esports Management Platform:** Tools for organizing and managing esports tournaments and leagues.  \n**Unique Selling Propositions (USPs) & Key Differentiators:**  \n- Advanced matchmaking algorithms that ensure fair and competitive play.  \n- Strong community engagement and support, fostering a loyal player base.  \n**Business Model:** B2C with a focus on free-to-play games that offer in-game purchases and premium content.  \n**Technology Used:** Proprietary game engines optimized for high-performance multiplay

In [187]:
pos_for_2

['**Company Name:** PixelCraft Studios  \n**Website:** pixelcraftstudios.com  \n**Industry:** Specializes in developing advanced graphics engines and tools for video game developers.  \n**Target Audience:** Video game development companies, indie game studios, and interactive entertainment firms.  \n**Products & Services:**  \n- **PixelEngine Pro:** A high-performance graphics engine that supports real-time rendering and advanced visual effects.  \n- **GameDev Toolkit:** A suite of tools designed to streamline the game development process, including asset management and debugging utilities.  \n**Unique Selling Propositions (USPs) & Key Differentiators:**  \n- Cutting-edge graphics technology that enhances visual fidelity and performance.  \n- Comprehensive support and integration with popular game development platforms like Unity and Unreal Engine.  \n**Business Model:** B2B with licensing and support subscription services.  \n**Technology Used:** Real-time rendering, advanced shaders,

In [197]:
!pwd

/home/ec2-user/SageMaker/ColBERT


In [199]:
import os
dirs=os.walk('./experiments')

In [201]:
[d for d in dirs]

[('./experiments',
  ['run_1741616267_evaluation',
   'colbert_aspect_training',
   'run_1741558706_evaluation',
   'colbert-ir',
   'run_1741795018_evaluation',
   '.ipynb_checkpoints'],
  []),
 ('./experiments/run_1741616267_evaluation',
  ['indexes', 'colbert_searcher'],
  []),
 ('./experiments/run_1741616267_evaluation/indexes',
  ['run_1741616267.test.nbits=2',
   'run_1741616267.colbert.best.all.nbits=2',
   'run_1741616267.val.nbits=2'],
  []),
 ('./experiments/run_1741616267_evaluation/indexes/run_1741616267.test.nbits=2',
  [],
  ['buckets.pt', 'plan.json', 'centroids.pt', 'avg_residual.pt']),
 ('./experiments/run_1741616267_evaluation/indexes/run_1741616267.colbert.best.all.nbits=2',
  [],
  ['buckets.pt',
   'doclens.0.json',
   '0.codes.pt',
   '2.metadata.json',
   'metadata.json',
   '2.codes.pt',
   '0.metadata.json',
   'doclens.2.json',
   '0.residuals.pt',
   'plan.json',
   '1.codes.pt',
   'ivf.pid.pt',
   'doclens.1.json',
   'centroids.pt',
   '1.residuals.pt',
  

In [1]:


import sys
import os

# Add the path to the parent directory
sys.path.append(os.path.abspath(".."))
from dep.azure_openai import StructuredResponseHandler,AzureOpenAIChatCompletion

from dep.response_formats import SyntheticPositiveFactsheetResponse

/home/ec2-user/anaconda3/envs/colbert_env/lib/python3.10/site-packages/pydantic/_internal/_config.py:345: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)


In [2]:
# s=StructuredResponseHandler(system_message='You need to return fatchseet of a company based on user query')

In [3]:
# res=s.structured_response(prompt='companies in AI sector',temperature=0.5,
#                       response_format=SyntheticPositiveFactsheetResponse, use_json= False)

In [4]:
client=AzureOpenAIChatCompletion()

In [5]:
client.get_competition('create a factsheet about a company in AI industry',response_format=SyntheticPositiveFactsheetResponse)

BadRequestError: Error code: 400 - {'error': {'code': 'BadRequest', 'message': 'response_format value as json_schema is enabled only for api versions 2024-08-01-preview and later'}}